In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier
import joblib


def metricas_classificacao(y_true, y_pred, target_names):
    report = classification_report(
        y_true,
        y_pred,
        target_names=target_names,
        output_dict=True,
        zero_division=0
    )
    tabela = pd.DataFrame(report).transpose()
    tabela = tabela.drop(index='accuracy', errors='ignore')
    tabela = tabela[['precision', 'recall', 'f1-score', 'support']]
    tabela['support'] = tabela['support'].astype(int)
    return tabela

# 1. Carga e limpeza
df_text = pd.read_csv('../data/Mental Health Disorder Detection Dataset.csv')
df_text = df_text.dropna(subset=['body', 'category'])

# 2. Preparacao do alvo
le_nlp = LabelEncoder()
y = le_nlp.fit_transform(df_text['category'])
target_names = le_nlp.classes_

print('Distribuicao das categorias (sem reamostragem):')
print(pd.Series(le_nlp.inverse_transform(y)).value_counts().sort_index())

# 3. Divisao estratificada
X_train_t, X_test_t, y_train_t, y_test_t = train_test_split(
    df_text['body'], y, test_size=0.2, random_state=42, stratify=y
)

# 4. Baseline TF-IDF
print('Vetorizando textos com TF-IDF baseline...')
vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=5
)

X_train_vec = vectorizer.fit_transform(X_train_t)
X_test_vec = vectorizer.transform(X_test_t)

# 5. Tratamento de desbalanceamento moderado sem reamostragem sintetica
# Em texto, exemplos sinteticos podem distorcer o espaco TF-IDF.
# O XGBoost recebe pesos balanceados para compensar diferencas de frequencia.
sample_weight = compute_sample_weight(class_weight='balanced', y=y_train_t)

# 6. Baseline XGBoost com TF-IDF
print('Treinando baseline XGBoost com TF-IDF...')
modelo_nlp_final = XGBClassifier(
    n_estimators=250,
    learning_rate=0.05,
    max_depth=6,
    random_state=42,
    eval_metric='mlogloss'
)
modelo_nlp_final.fit(X_train_vec, y_train_t, sample_weight=sample_weight)

# 7. Predicao e avaliacao
y_pred_t = modelo_nlp_final.predict(X_test_vec)

print('\n=== RELATORIO DE CLASSIFICACAO NLP ===')
print(metricas_classificacao(y_test_t, y_pred_t, target_names).to_string(float_format=lambda value: f'{value:.4f}'))

# 8. Salvando os arquivos definitivos
joblib.dump(modelo_nlp_final, '../src/modelo_nlp_final.pkl')
joblib.dump(vectorizer, '../src/vectorizer_nlp.pkl')
joblib.dump(le_nlp, '../src/label_encoder_nlp.pkl')
print('\nModelos salvos com sucesso na pasta /src/.')

# 9. Matriz de Confusao
plt.figure(figsize=(12, 8))
sns.heatmap(
    confusion_matrix(y_test_t, y_pred_t),
    annot=True,
    fmt='d',
    xticklabels=target_names,
    yticklabels=target_names,
    cmap='YlGnBu'
)
plt.title('Matriz de Confusao - Baseline TF-IDF')
plt.show()

# 10. Funcao de teste
def prever_texto(texto):
    vec = vectorizer.transform([texto])
    pred = modelo_nlp_final.predict(vec)[0]
    prob = modelo_nlp_final.predict_proba(vec)[0][pred]
    doenca = le_nlp.inverse_transform([pred])[0]
    return doenca, prob

texto_teste = "I've been feeling very anxious and my heart is always racing when I think about work."
doenca, certeza = prever_texto(texto_teste)
print(f'\nTeste pratico:\nTexto: {texto_teste}\nPredicao: {doenca} ({certeza*100:.2f}% de confianca)')
